In [1]:
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import ContextualCompressionRetriever

C:\Users\cored\AppData\Local\Temp\ipykernel_17748\2128608982.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [3]:
# Dummy documents covering different topics, each with a mix of relevant and tangential info
docs = [
    Document(
        page_content=(
            "Artificial intelligence has made remarkable strides in natural language processing, "
            "with large language models now capable of generating human-quality text and code. "
            "Computer vision systems can identify objects in images with superhuman accuracy, "
            "powering applications from autonomous vehicles to medical imaging diagnostics. "
            "However, the rapid advancement of AI has raised significant ethical concerns about "
            "job displacement, algorithmic bias, and the concentration of power among a few tech companies."
        ),
        metadata={"topic": "artificial_intelligence"},
    ),
    Document(
        page_content=(
            "Global temperatures have risen by approximately 1.1 degrees Celsius since pre-industrial "
            "times, driven primarily by the burning of fossil fuels. The melting of polar ice caps has "
            "accelerated, contributing to rising sea levels that threaten coastal communities worldwide. "
            "Renewable energy adoption is growing rapidly, with solar and wind power becoming cheaper "
            "than coal in many regions. Governments are implementing carbon pricing mechanisms and "
            "investing in green infrastructure to meet Paris Agreement targets."
        ),
        metadata={"topic": "climate_change"},
    ),
    Document(
        page_content=(
            "NASA's Artemis program aims to return humans to the Moon by the mid-2020s, establishing "
            "a sustainable presence as a stepping stone to Mars. Private companies like SpaceX are "
            "developing reusable rocket technology that has dramatically reduced launch costs. "
            "The James Webb Space Telescope has captured unprecedented images of distant galaxies, "
            "revealing new insights about the early universe. Asteroid mining is being explored as a "
            "potential source of rare minerals needed for electronics manufacturing."
        ),
        metadata={"topic": "space_exploration"},
    ),
    Document(
        page_content=(
            "CRISPR gene editing technology has revolutionized medical genomics, enabling precise "
            "modifications to DNA sequences that were previously impossible. Researchers are using "
            "genomic data to develop personalized medicine approaches, tailoring treatments based on "
            "an individual's genetic profile. Recent breakthroughs in mRNA technology, accelerated by "
            "COVID-19 vaccine development, are now being applied to cancer immunotherapy and rare "
            "genetic disorders. Hospital information systems are increasingly integrating genomic data "
            "to support clinical decision-making at the point of care."
        ),
        metadata={"topic": "medicine"},
    ),
    Document(
        page_content=(
            "The global economy is navigating a period of high inflation driven by supply chain "
            "disruptions, energy price volatility, and post-pandemic demand surges. Central banks "
            "worldwide have raised interest rates aggressively to combat inflation, impacting housing "
            "markets and consumer spending. Cryptocurrency regulation is becoming a priority for "
            "financial authorities, with the EU's MiCA framework setting a global precedent. "
            "Trade tensions between major economies continue to reshape global supply chains, "
            "pushing companies toward nearshoring and diversification strategies."
        ),
        metadata={"topic": "economics"},
    ),
    Document(
        page_content=(
            "Quantum computing has reached a critical milestone with several companies demonstrating "
            "quantum advantage on specific computational tasks. Error correction remains the biggest "
            "challenge, as current quantum processors are highly susceptible to noise and decoherence. "
            "Quantum simulation of molecular structures could transform drug discovery by accurately "
            "modeling protein folding and chemical interactions. Major tech companies and governments "
            "are investing billions in quantum research, viewing it as essential for national security "
            "and economic competitiveness."
        ),
        metadata={"topic": "quantum_computing"},
    ),
]

print(f"Created {len(docs)} documents")

Created 6 documents


In [4]:
print(docs[0].page_content)

Artificial intelligence has made remarkable strides in natural language processing, with large language models now capable of generating human-quality text and code. Computer vision systems can identify objects in images with superhuman accuracy, powering applications from autonomous vehicles to medical imaging diagnostics. However, the rapid advancement of AI has raised significant ethical concerns about job displacement, algorithmic bias, and the concentration of power among a few tech companies.


In [5]:
embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")

In [ ]:
## dense retriver :qwen
Embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")

NameError: name 'InMemoryVectorStore' is not defined

In [15]:
## store in vectorstore db
vectorstore = Chroma.from_documents(docs, Embeddings, persist_directory="./chroma_db")

In [ ]:
chroma_retriver = vectorstore.as_retriever(search_kwargs={"k": 3},search_type="similarity")

# BM25Plus ensures every matched term contributes a positive score,
# which improves recall for short documents like the ones we have here used for short docs
bm25_retriever = BM25Retriever.from_documents(docs, k=3, bm25_variant="Plus") 

In [25]:
# EnsembleRetriever merges results from both retrievers using Reciprocal Rank Fusion (RRF)
ensemble_retiver = EnsembleRetriever(retrievers=[chroma_retriver, bm25_retriever],k=3, weights=[0.8,0.2])


In [27]:
query = "How do vaccines work?"
bm25_results = bm25_retriever.invoke(query)
chroma_results = chroma_retriver.invoke(query)
ensemble_results = ensemble_retiver.invoke(query)[:3]

In [28]:
# BM25 matches on the exact word "vaccine" — finds docs 1 and 2
# but misses the semantically related immune/antibody docs (3, 4, 5)
print("=== BM25 Only (keyword match) ===")
for i, doc in enumerate(bm25_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

print()

# Dense search finds docs 3, 4, 5 through semantic understanding
# even though they don't contain the word "vaccine"
print("=== ChromaDB Only (semantic match) ===")
for i, doc in enumerate(chroma_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

print()

# Ensemble combines both — surfaces keyword matches AND semantic matches
print("=== Ensemble / Hybrid (keyword + semantic) ===")
for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")

=== BM25 Only (keyword match) ===
  [1] topic=programming: REST APIs communicate over HTTP using standard methods like GET, POST, PUT, and DELETE.
  [2] topic=nature: Coral reefs cover less than 1% of the ocean floor but support about 25% of all marine species.
  [3] topic=nature: The Amazon rainforest produces about 20% of the world's oxygen and houses 10% of all species.

=== ChromaDB Only (semantic match) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [3] topic=health: The flu vaccine is reformulated each year to match the most prevalent circulating virus strains.

=== Ensemble / Hybrid (keyword + semantic) ===
  [1] topic=health: Vaccines work by introducing a weakened or inactivated pathogen to trigger an immune response.
  [2] topic=health: The flu vaccine is reformulated each year to match the